# DPO Training (LoRA -> DPO)

In [1]:
# Run this on Remote Jupyter Book.

import os

os.getcwd()
os.chdir("/root/OtakuLab")

## Configure Input Constants

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path
from typing import Any, Optional, cast

import torch

RAW_PAIR_JSONL_PATH = Path("./data/dpo_pairs_v41_rs.jsonl")
FORMATTED_DPO_JSONL_PATH = Path("./data/dpo_train_v41.jsonl")

BASE_MODEL_PATH = Path("/root/autodl-tmp/Qwen3-1.7B/")
V41_LORA_PATH = Path("/root/OtakuLab/outputs/styled-qwen-v4.3-4e-05/lora")
# BASE_MODEL_PATH = Path("../Models/Qwen3-1.7B/")
# V41_LORA_PATH = Path("./outputs/model/styled-qwen-v4.1-4e-05/lora")
DPO_OUTPUT_DIR = Path("./outputs/styled-qwen-dpo-v2")

STRICT_REQUIRE_THINK = True
MAX_FORMAT_SAMPLES: Optional[int] = None

print("Raw pair JSONL:", RAW_PAIR_JSONL_PATH)
print("Output JSONL  :", FORMATTED_DPO_JSONL_PATH)
print("Base model    :", BASE_MODEL_PATH)
print("V4.3 LoRA     :", V41_LORA_PATH)
print("DPO output dir:", DPO_OUTPUT_DIR)

/root/miniconda3/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Raw pair JSON : data/dpo_pairs_v41_rs.json
Raw pair JSONL: data/dpo_pairs_v41_rs.jsonl
Output JSON   : data/dpo_train_v41.json
Output JSONL  : data/dpo_train_v41.jsonl
Base model    : /root/autodl-tmp/Qwen3-1.7B
V4.3 LoRA     : /root/OtakuLab/outputs/styled-qwen-v4.3-4e-05/lora
DPO output dir: outputs/styled-qwen-dpo-v2


## Format Dataset into DPO JSONL

In [ ]:
from transformers import AutoTokenizer

SYSTEM_PROMPT = (
    "You are an expert in stylistic rewriting. "
    "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
)

THINK_BLOCK_PATTERN = re.compile(r"<think>.*?</think>", re.DOTALL)

def load_raw_pair_rows() -> list[dict[str, Any]]:
    if RAW_PAIR_JSONL_PATH.exists():
        rows: list[dict[str, Any]] = []
        for line in RAW_PAIR_JSONL_PATH.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if isinstance(obj, dict):
                rows.append(obj)
        return rows

    raise FileNotFoundError(
        f"No input pair file found. Checked: {RAW_PAIR_JSONL_PATH}"
    )

def build_user_prompt_from_row(row: dict[str, Any]) -> Optional[str]:
    character = str(row.get("character", "")).strip()
    neutral_sentence = str(row.get("neutral_sentence", "")).strip()
    instr = row.get("instruction_components", {})

    if not isinstance(instr, dict):
        instr = {}

    semantic_tags = instr.get("semantic_tags", [])
    signature_rules = instr.get("signature_rules", [])

    if not isinstance(semantic_tags, list):
        semantic_tags = []
    if not isinstance(signature_rules, list):
        signature_rules = []

    if not character or not neutral_sentence:
        return None

    semantic_text = ", ".join(str(x) for x in semantic_tags) if semantic_tags else "None"
    signature_text = " ".join(str(x) for x in signature_rules) if signature_rules else "None"

    return (
        f"Target Character: {character}\n"
        f"Semantic Traits: {semantic_text}\n"
        f"Formatting Signature: {signature_text}\n"
        f"Neutral Content: {neutral_sentence}\n"
    )

def to_prompt_string(row: dict[str, Any], tokenizer: Any) -> Optional[str]:
    prompt_obj = row.get("prompt")

    if isinstance(prompt_obj, str) and "<|im_start|>assistant" in prompt_obj:
        return prompt_obj

    if isinstance(prompt_obj, list):
        return tokenizer.apply_chat_template(
            prompt_obj,
            tokenize=False,
            add_generation_prompt=True,
        )

    user_prompt = build_user_prompt_from_row(row)
    if user_prompt is None:
        return None

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def normalize_assistant_text(text: str) -> tuple[str, Optional[str]]:
    cleaned = (text or "").strip()
    if not cleaned:
        return "", "empty_text"

    has_think = bool(THINK_BLOCK_PATTERN.search(cleaned))
    if STRICT_REQUIRE_THINK and not has_think:
        return "", "missing_think_block"

    if not cleaned.endswith("<|im_end|>"):
        cleaned = cleaned + "<|im_end|>"

    return cleaned, None

In [ ]:
if not BASE_MODEL_PATH.exists():
    raise FileNotFoundError(f"Base model path does not exist: {BASE_MODEL_PATH}")

tokenizer_for_format = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
raw_rows = load_raw_pair_rows()
if MAX_FORMAT_SAMPLES is not None:
    raw_rows = raw_rows[:MAX_FORMAT_SAMPLES]

formatted_rows: list[dict[str, str]] = []
skip_counter: Counter[str] = Counter()

for row in raw_rows:
    prompt_text = to_prompt_string(row, tokenizer_for_format)
    if not prompt_text:
        skip_counter["missing_prompt"] += 1
        continue

    chosen_text, chosen_err = normalize_assistant_text(str(row.get("chosen", "")))
    if chosen_err is not None:
        skip_counter[f"chosen:{chosen_err}"] += 1
        continue

    rejected_text, rejected_err = normalize_assistant_text(str(row.get("rejected", "")))
    if rejected_err is not None:
        skip_counter[f"rejected:{rejected_err}"] += 1
        continue
        
    # Share CoT: replace rejected's think block with chosen's think block
    chosen_think_match = THINK_BLOCK_PATTERN.search(chosen_text)
    if chosen_think_match:
        rejected_text = THINK_BLOCK_PATTERN.sub(chosen_think_match.group(0), rejected_text, count=1)

    if chosen_text == rejected_text:
        skip_counter["chosen_equals_rejected"] += 1
        continue

    length_ratio = len(chosen_text) / len(rejected_text)

    if length_ratio < 0.6 or length_ratio > 1.5:
        skip_counter["length_ratio_outlier"] += 1
        continue

    formatted_rows.append(
        {
            "prompt": prompt_text,
            "chosen": chosen_text,
            "rejected": rejected_text,
        }
    )

with FORMATTED_DPO_JSONL_PATH.open("w", encoding="utf-8") as f:
    for item in formatted_rows:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Raw rows       :", len(raw_rows))
print("Kept rows      :", len(formatted_rows))
print("Dropped rows   :", sum(skip_counter.values()))
print("Drop reasons   :", dict(skip_counter))
print("Saved JSONL    :", FORMATTED_DPO_JSONL_PATH)

if formatted_rows:
    sample = formatted_rows[0]
    print("=== Sample Preview ===")
    print("Prompt tail:", sample["prompt"][-180:])
    print("Chosen tail:", sample["chosen"][-180:])
    print("Rejected tail:", sample["rejected"][-180:])

Raw rows       : 8272
Kept rows      : 8249
Dropped rows   : 23
Drop reasons   : {'chosen_equals_rejected': 4, 'length_ratio_outlier': 19}
Saved JSON     : data/dpo_train_v41.json
Saved JSONL    : data/dpo_train_v41.jsonl
=== Sample Preview ===
Prompt tail: rt|>user
Target Character: 沐雪
Semantic Traits: airhead, naive, humorous, playful, cute
Formatting Signature: None
Neutral Content: “奇奇怪怪的东西”指的是什么？
<|im_end|>
<|im_start|>assistant

Chosen tail: <think>她依天真语义选择发问，极高碎片率与疑问率使句式松散，最后用“这个嘛”及省略号、括号动作展现呆萌可爱的身份特征。</think>

这个嘛...奇奇怪怪的东西指的是什么呢（天真）<|im_end|>
Rejected tail: <think>她依天真语义选择发问，极高碎片率与疑问率使句式松散，最后用“这个嘛”及省略号、括号动作展现呆萌可爱的身份特征。</think>

“奇怪的东西”指的是什么？<|im_end|>


## Train DPO with TRL

### Configure DPO Config

In [ ]:
import inspect

from datasets import load_dataset
from peft import PeftModel
from transformers import AutoTokenizer, Qwen3ForCausalLM
from trl.trainer.dpo_config import DPOConfig
from trl.trainer.dpo_trainer import DPOTrainer

dpo_config = DPOConfig(
    output_dir=str(DPO_OUTPUT_DIR),
    beta=0.1,
    learning_rate=1e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    max_length=768,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    bf16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    report_to="none",
)

if not FORMATTED_DPO_JSONL_PATH.exists():
    raise FileNotFoundError(f"Missing formatted dataset: {FORMATTED_DPO_JSONL_PATH}")
if not V41_LORA_PATH.exists():
    raise FileNotFoundError(f"Missing V4.3 LoRA path: {V41_LORA_PATH}")

train_dataset = load_dataset("json", data_files=str(FORMATTED_DPO_JSONL_PATH), split="train")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
base_model = Qwen3ForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)

model = PeftModel.from_pretrained(base_model, V41_LORA_PATH, is_trainable=True)
model_config = cast(Any, getattr(model, "config", None))
if model_config is not None:
    model_config.use_cache = False

print(train_dataset)
model.print_trainable_parameters()

Generating train split: 0 examples [00:00, ? examples/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 8249
})
trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


### (Not Recommended) Final-Response-Only DPO

keep `<think>` as context, but optimize only the final response tokens after `</think>`.

**Since this method produces poor results, it is not recommended.**

set `TRAIN_ONLY_FINAL_RESPONSE = True` to enable its feature.

In [ ]:
from transformers import TrainerCallback

TRAIN_ONLY_FINAL_RESPONSE = False
THINK_END_TAG = "</think>"
THINK_END_TOKEN_IDS = tokenizer(THINK_END_TAG, add_special_tokens=False)["input_ids"]
if TRAIN_ONLY_FINAL_RESPONSE and not THINK_END_TOKEN_IDS:
    raise ValueError("Tokenizer could not encode </think>; cannot enable final-response-only masking.")


def find_subsequence(sequence: list[int], pattern: list[int]) -> int:
    if not pattern or len(pattern) > len(sequence):
        return -1

    last_start = len(sequence) - len(pattern) + 1
    for start in range(last_start):
        if sequence[start : start + len(pattern)] == pattern:
            return start

    return -1


class FinalResponseOnlyMaskCollator:
    def __init__(self, base_collator: Any, think_end_token_ids: list[int]):
        self.base_collator = base_collator
        self.think_end_token_ids = think_end_token_ids

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, Any]:
        batch = self.base_collator(features)
        input_ids = batch.get("input_ids")
        completion_mask = batch.get("completion_mask")

        if input_ids is None or completion_mask is None:
            return batch

        # Keep only final answer tokens trainable: mask out completion tokens up to </think>.
        for row_idx in range(input_ids.shape[0]):
            row_ids = input_ids[row_idx]
            row_mask = completion_mask[row_idx]

            active_positions = (row_mask == 1).nonzero(as_tuple=False).flatten()
            if active_positions.numel() == 0:
                continue

            comp_start = int(active_positions[0].item())
            comp_end = int(active_positions[-1].item()) + 1

            completion_ids = row_ids[comp_start:comp_end].tolist()
            local_think_end = find_subsequence(completion_ids, self.think_end_token_ids)
            if local_think_end < 0:
                continue

            global_cutoff = comp_start + local_think_end + len(self.think_end_token_ids)
            row_mask[comp_start:global_cutoff] = 0

        batch["completion_mask"] = completion_mask
        return batch


class DPOMetricPrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        step = state.global_step
        acc = logs.get("rewards/accuracies", None)
        margin = logs.get("rewards/margins", None)
        chosen = logs.get("rewards/chosen", None)
        rejected = logs.get("rewards/rejected", None)

        if acc is not None:
            print(f"Step {step:3d} | Acc: {acc:.4f} | Margin: {margin:.4f} | Chosen: {chosen:.4f} | Rejected: {rejected:.4f}")


trainer_sig = inspect.signature(DPOTrainer.__init__)
trainer_kwargs: dict[str, Any] = {
    "model": model,
    "args": dpo_config,
    "train_dataset": train_dataset,
    "callbacks": [DPOMetricPrinterCallback()],
}

if "ref_model" in trainer_sig.parameters:
    trainer_kwargs["ref_model"] = None

if "tokenizer" in trainer_sig.parameters:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in trainer_sig.parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    raise RuntimeError("Cannot find tokenizer/processing_class argument in DPOTrainer signature")

trainer = DPOTrainer(**trainer_kwargs)

if TRAIN_ONLY_FINAL_RESPONSE:
    trainer.data_collator = FinalResponseOnlyMaskCollator(
        cast(Any, trainer.data_collator),
        THINK_END_TOKEN_IDS,
    )
    print("Final-response-only mask enabled: completion tokens before </think> are excluded from loss.")

# 验证 reference 分支确实 frozen
with trainer.model.disable_adapter():  # type: ignore[attr-defined]
    params_requiring_grad = sum(
        p.requires_grad for p in trainer.model.parameters()  # type: ignore[attr-defined]
    )
assert params_requiring_grad == 0, "Reference branch is not frozen!"

print("DPOTrainer initialized.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/8249 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8249 [00:00<?, ? examples/s]

Final-response-only mask enabled: completion tokens before </think> are excluded from loss.
DPOTrainer initialized.


### Start Training And Export Final Model

In [8]:
DPO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_result = trainer.train()
metrics = train_result.metrics

save_dir = DPO_OUTPUT_DIR / "lora"
trainer.save_model(str(save_dir))
tokenizer.save_pretrained(DPO_OUTPUT_DIR / "tokenizer")

metrics_path = DPO_OUTPUT_DIR / "train_metrics.json"
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

if model_config is not None:
    model_config.use_cache = True

print("Training complete.")
print("Saved LoRA   :", save_dir)
print("Saved tokenizer:", DPO_OUTPUT_DIR / "tokenizer")
print("Saved metrics :", metrics_path)
print("Train metrics :", metrics)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.690271
20,0.691298
30,0.679400
40,0.672509
50,0.656989
60,0.646454
70,0.617433
80,0.602504
90,0.553822
100,0.577696


Step  10 | Acc: 0.4375 | Margin: 0.0061 | Chosen: 0.0027 | Rejected: -0.0034
Step  20 | Acc: 0.5750 | Margin: 0.0040 | Chosen: 0.0011 | Rejected: -0.0029
Step  30 | Acc: 0.7125 | Margin: 0.0282 | Chosen: 0.0045 | Rejected: -0.0237
Step  40 | Acc: 0.7625 | Margin: 0.0426 | Chosen: -0.0057 | Rejected: -0.0483
Step  50 | Acc: 0.8875 | Margin: 0.0749 | Chosen: -0.0165 | Rejected: -0.0914
Step  60 | Acc: 0.8875 | Margin: 0.0986 | Chosen: -0.0319 | Rejected: -0.1305
Step  70 | Acc: 0.9000 | Margin: 0.1616 | Chosen: -0.0281 | Rejected: -0.1896
Step  80 | Acc: 0.8750 | Margin: 0.2019 | Chosen: -0.0716 | Rejected: -0.2735
Step  90 | Acc: 0.9625 | Margin: 0.3170 | Chosen: -0.0778 | Rejected: -0.3948
Step 100 | Acc: 0.8875 | Margin: 0.2627 | Chosen: -0.1444 | Rejected: -0.4070
Step 110 | Acc: 0.9000 | Margin: 0.4748 | Chosen: -0.2123 | Rejected: -0.6871
Step 120 | Acc: 0.9625 | Margin: 0.5633 | Chosen: -0.2380 | Rejected: -0.8014
Step 130 | Acc: 0.9500 | Margin: 0.6313 | Chosen: -0.3045 | Rejecte